# Thử giảm WER PhoWhisper — 2×T4
Giữ model 13,91% làm mốc. Chạy decoding trên validation cũ trước; train thử là bước tùy chọn, xuất sang thư mục riêng. Không bảo đảm đạt 8%. Nếu phiên Kaggle cũ đã hết, cần gắn Output đã lưu chứa model và split_manifest.json rồi sửa các đường dẫn bên dưới. Không tải model Drive cho thử nghiệm này.

In [ ]:
%pip install -q "transformers==4.51.3" "peft==0.15.2" "accelerate>=1.6,<2" soundfile scipy numpy

In [ ]:
from pathlib import Path
import os, sys, subprocess
TRAIN_JSON = Path('/kaggle/input/datasets/n107hoangtuong/zalo-competion/train+public_test/train/train.jsonl')
AUDIO_ROOT = TRAIN_JSON.parent
WORK = Path('/kaggle/working/lifemate-asr')
PUBLIC_REPO = 'https://github.com/tuong1901/zac2026-lifemate.git'
PUBLIC_BRANCH = 'feat-eval-LLM'
if not (WORK / '.git').is_dir():
    assert not WORK.exists() or not any(WORK.iterdir()), 'Choose an empty WORK directory for git clone'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', PUBLIC_BRANCH,
                    PUBLIC_REPO, str(WORK)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(WORK), 'remote', 'get-url', 'origin'], text=True).strip()
    assert remote == PUBLIC_REPO, f'Unexpected origin: {remote}'
    branch = subprocess.check_output(['git', '-C', str(WORK), 'branch', '--show-current'], text=True).strip()
    assert branch == PUBLIC_BRANCH, f'Checkout {PUBLIC_BRANCH} before updating'
    changes = subprocess.check_output(['git', '-C', str(WORK), 'status', '--porcelain'], text=True)
    if changes.strip():
        print('Local edits found; keeping them. Commit or stash them manually before pulling updates.')
        print(changes)
    else:
        subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only', 'origin', PUBLIC_BRANCH], check=True)
assert (WORK / 'src/asr/finetune.py').is_file()
os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
import torch
assert torch.cuda.device_count() == 2, 'Select T4 x2 and restart session before importing torch'
assert TRAIN_JSON.is_file(), f'Check TRAIN_JSON: {TRAIN_JSON}'
assert (AUDIO_ROOT / 'audio').is_dir(), f'Check AUDIO_ROOT: {AUDIO_ROOT}'
print('GPU:', torch.cuda.get_device_name(0))
print('Code:', WORK / 'src/asr')
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Training manifest:', TRAIN_JSON)
# After updating code already imported in this kernel, restart the session before importing again.
# The train/merge subprocesses always read the current files on disk.

In [ ]:
import json
BASE_RUN = Path('/kaggle/working/asr_training')
BASE_MODEL = Path('/kaggle/working/phowhisper-large-finetuned')
SPLIT = BASE_RUN / 'split_manifest.json'
EXPERIMENTS = Path('/kaggle/working/asr_wer_experiments')
assert SPLIT.is_file(), 'Attach the old run outputs or restore the prior Kaggle session'
assert (BASE_MODEL / 'config.json').is_file(), 'Point BASE_MODEL to the merged 13.91% model'
if (BASE_RUN / 'baseline_metrics.json').is_file():
    print('Original base WER:', json.loads((BASE_RUN / 'baseline_metrics.json').read_text()))
for cp in sorted(BASE_RUN.glob('checkpoint-*')):
    state_file = cp / 'trainer_state.json'
    if state_file.is_file():
        state = json.loads(state_file.read_text())
        print(cp.name, [(r.get('epoch'), r.get('eval_wer')) for r in state.get('log_history', []) if 'eval_wer' in r])

In [ ]:
# Two workers split validation audio. Each evaluates the same three decoding settings.
# This can take substantial GPU time; no weights are trained.
DECODE_OUT = EXPERIMENTS / 'decode_existing'
subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone',
    '--nproc_per_node=2', '-m', 'src.asr.benchmark',
    '--manifest', str(SPLIT), '--model', str(BASE_MODEL),
    '--out', str(DECODE_OUT), '--configs', '1:3,1:0,5:0'], check=True)

In [ ]:
report = json.loads((DECODE_OUT / 'report.json').read_text())
print(json.dumps(report, indent=2))
# WER 0.10 = 10%. Beam5 may improve or worsen WER; keep only measured improvements.
# This tunes decoding on validation. Reserve a separate untouched test for final claims.
from src.asr.evaluate import compute_wer
best = report['best_config']
with (DECODE_OUT / f'{best}.predictions.jsonl').open() as stream:
    predictions = [json.loads(line) for line in stream if line.strip()]
for row in sorted(predictions, key=lambda r: compute_wer(r['reference'], r['prediction'])['wer'], reverse=True)[:15]:
    print('REF:', row['reference'])
    print('HYP:', row['prediction'], '\n')

Đặt RUN_TRAIN=True khi sẵn sàng. Mặc định warm_start dùng adapter cũ, optimizer mới, LR=2e-5 và học thêm 1 epoch với speed perturbation trên split cũ. Đây không phải resume optimizer cũ. speed_only và reference_like vẫn là các thử nghiệm khởi tạo từ base, 3 epochs. Chỉ giữ model mới nếu validation WER tốt hơn model cũ; không bảo đảm học thêm sẽ cải thiện. Output mới giữ riêng, không ghi đè checkpoint/model cũ.

In [ ]:
RUN_TRAIN = False
PROFILE = 'warm_start'  # reuse the best old adapter; alternatives: speed_only, reference_like
profiles = {
    'warm_start': {'lr': '2e-5', 'accum': '16', 'epochs': '1'},
    'speed_only': {'lr': '1e-4', 'accum': '16', 'epochs': '3'},
    'reference_like': {'lr': '5e-4', 'accum': '8', 'epochs': '3'},
}
assert PROFILE in profiles
NEW_RUN = EXPERIMENTS / PROFILE
settings = profiles[PROFILE]
if RUN_TRAIN:
    command = [sys.executable, '-m', 'torch.distributed.run', '--standalone',
        '--nproc_per_node=2', '-m', 'src.asr.finetune', 'train',
        '--train-json', str(TRAIN_JSON), '--audio-root', str(AUDIO_ROOT),
        '--split-manifest', str(SPLIT), '--model', 'vinai/PhoWhisper-large',
        '--out', str(NEW_RUN), '--epochs', settings['epochs'], '--batch-size', '1',
        '--grad-accum', settings['accum'], '--lr', settings['lr'],
        '--rank', '32', '--specaug', '0.05', '--speed', '0.1', '--seed', '42', '--eval-beams', '5', '--eval-ngram', '0']
    if PROFILE == 'warm_start':
        import importlib
        training_data = importlib.reload(importlib.import_module('src.asr.training_data'))
        initial_adapter, initial_metrics = training_data.select_saved_checkpoint(BASE_RUN)
        print('Reusing trained adapter:', initial_adapter, initial_metrics)
        command += ['--init-adapter', str(initial_adapter)]
    subprocess.run(command, check=True)
else:
    print('Training disabled. Review decoding first, then set RUN_TRAIN=True.')

In [ ]:
if RUN_TRAIN:
    print(json.loads((NEW_RUN / 'validation_metrics.json').read_text()))
    print(json.loads((NEW_RUN / 'selected_checkpoint.json').read_text()))
    NEW_MODEL = EXPERIMENTS / f'{PROFILE}_merged'
    subprocess.run([sys.executable, '-m', 'src.asr.finetune', 'merge',
        '--adapter', str(NEW_RUN / 'best_adapter'), '--out', str(NEW_MODEL)], check=True)

In [ ]:
if RUN_TRAIN:
    NEW_DECODE_OUT = EXPERIMENTS / f'decode_{PROFILE}'
    subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone',
        '--nproc_per_node=2', '-m', 'src.asr.benchmark',
        '--manifest', str(SPLIT), '--model', str(NEW_MODEL),
        '--out', str(NEW_DECODE_OUT), '--configs', '1:3,1:0,5:0'], check=True)
    newer = json.loads((NEW_DECODE_OUT / 'report.json').read_text())
    assert newer['manifest_sha256'] == report['manifest_sha256']
    for config in report['scores']:
        before, after = report['scores'][config]['wer'], newer['scores'][config]['wer']
        print(config, f'old={before:.2%}, new={after:.2%}, difference={(after-before)*100:+.2f} percentage points')